In [1]:
import numpy as np
from scipy.stats import f, chi2, norm
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold

In [2]:
def center(X, mean_vec=None):
    if mean_vec is None:
        mean_vec = X.mean(axis=0)
    return X - mean_vec, mean_vec

def compute_residuals_and_scores(X, pca, X_mean=None):
    # X: (n_samples, n_features) - assumed centered if X_mean provided will center
    if X_mean is None:
        Xc = X - X.mean(axis=0)
    else:
        Xc = X - X_mean
    T = pca.transform(Xc)                    # scores (n x A)
    P = pca.components_.T                    # loadings (J x A)
    E = Xc - T.dot(P.T)                      # residuals (n x J)
    eigvals = pca.explained_variance_        # length A
    return T, P, E, eigvals

def score_distance(T, eigvals):
    # T shape (n, A), eigvals length A
    eig = np.copy(eigvals)
    eig[eig <= 0] = 1e-12
    return np.sum((T ** 2) / eig, axis=1)

def orthogonal_distance(E):
    # E: residuals (n x J)
    return np.sum(E ** 2, axis=1)

# Jackson-Mudholkar approx for ODcrit
def jackson_mudholkar_od_threshold(eigvals_all, A, alpha=0.05):
    # eigvals_all: array of all eigenvalues (descending), A = number of PCs used
    # uses theta1, theta2, theta3 computed on residual eigenvalues (a = A+1..rank)
    lambdas = np.asarray(eigvals_all)
    if lambdas.ndim != 1:
        lambdas = lambdas.flatten()
    residual = lambdas[A:]
    if residual.size == 0:
        # no residual variance -> threshold 0 (degenerate)
        return 0.0
    theta1 = np.sum(residual)
    theta2 = np.sum(residual ** 2)
    theta3 = np.sum(residual ** 3)
    h0 = 1.0 - (2.0 * theta1 * theta3) / (3.0 * (theta2 ** 2) + 1e-18)
    # deviate for normal upper quantile
    c_inv = norm.ppf(1 - alpha)
    term = (2 * theta2 * h0 ** 2) / (theta1 ** 2 + 1e-18)
    inside = 1.0 + (c_inv * (term ** 0.5)) + (theta2 * h0 * (h0 - 1) / (theta1 ** 2 + 1e-18))
    # ensure positivity & numeric stability
    if inside <= 0:
        inside = 1.0
    ODcrit = (theta1) * (inside ** (1.0 / h0))
    return ODcrit

# Box's method for ODcrit: uses gB * chi2 quantile with hB df (approx)
def box_od_threshold(eigvals_all, A, alpha=0.05):
    residual = eigvals_all[A:]
    theta1 = np.sum(residual)
    theta2 = np.sum(residual ** 2)
    gB = theta2 / theta1 if theta1 != 0 else 1e-12
    hB = (theta1 ** 2) / theta2 if theta2 != 0 else 1.0
    crit = gB * chi2.ppf(1 - alpha, df=int(round(hB)))
    return float(crit)

def empirical_percentile_threshold(values, alpha=0.05):
    # (1-alpha) percentile
    q = np.percentile(values, 100 * (1 - alpha))
    return float(q)

# SD threshold options
def sd_threshold_hotelling(A, N, alpha=0.05):
    # SDcrit = A*(N^2 - 1)/(N*(N - A))*F^{-1}(alpha, A, N-A)
    if N - A <= 0:
        return np.inf
    F_val = f.ppf(1 - alpha, A, N - A)
    num = A * (N ** 2 - 1)
    den = N * (N - A)
    return (num / den) * F_val

def sd_threshold_chi2(A, alpha=0.05):
    # chi2 approximate with A dof
    return chi2.ppf(1 - alpha, df=A)

# DD-SIMCA fcrit (chi2 with LOD+LSD dof)
def dd_fcrit(OD0, SD0, s2_OD, s2_SD, alpha=0.05):
    LOD = (2 * (OD0 ** 2)) / (s2_OD + 1e-18)
    LSD = (2 * (SD0 ** 2)) / (s2_SD + 1e-18)
    return chi2.ppf(1 - alpha, df=int(round(LOD + LSD)))

# ------------------------
# SIMCA class model (per-class)
# ------------------------

class SIMCAClassModel:
    """
    A class model for a single class (fit only on samples of that class).
    Stores PCA, thresholds, and provides predict/diagnostics.
    """
    def __init__(self, n_components=None, alpha=0.05):
        self.n_components = n_components
        self.alpha = alpha
        self.pca = None
        self.mean_ = None
        # trained stats
        self.T_train = None
        self.P = None
        self.E_train = None
        self.eigvals = None
        self.OD_train = None
        self.SD_train = None
        self.thresholds = {}

    def fit(self, X):
        # X should be (n_samples_class, n_features)
        X = np.asarray(X)
        Xc, mean_vec = center(X)
        self.mean_ = mean_vec
        max_comp = min(Xc.shape[0] - 1, Xc.shape[1])
        # Never consume the full available rank: od_method='box'/'jackson' derive their
        # threshold from the residual eigenvalues, which are meaningless (pure floating-point
        # noise around zero) once components == rank, producing NaN/negative thresholds that
        # silently reject every future sample. Keep at least one genuine residual dimension.
        residual_safe_max = max_comp - 1
        n_comp = self.n_components if self.n_components is not None else max(1, min(max_comp, 10))
        n_comp = min(n_comp, residual_safe_max) if residual_safe_max >= 1 else 0
        if n_comp <= 0:
            raise ValueError("Not enough samples/features to fit PCA with a residual dimension remaining.")
        self.pca = PCA(n_components=n_comp)
        T = self.pca.fit_transform(Xc)
        P = self.pca.components_.T
        E = Xc - T.dot(P.T)
        eig = self.pca.explained_variance_
        self.T_train = T
        self.P = P
        self.E_train = E
        self.eigvals = eig
        self.OD_train = orthogonal_distance(E)
        self.SD_train = score_distance(T, eig)

    def set_thresholds(self,
               od_method='percentile',     # 'percentile', 'jackson', 'box'
               sd_method='f',              # 'f' or 'chi2' or 'percentile'
               dd_params=None):            # if using dd-simca, provide params dict
        """
        Compute and store thresholds for OD and SD according to chosen methods.
        dd_params: dict with keys (for DD-SIMCA) if needed.
        """
        # --- residual eigenvalues of the training data (fixed property of this
        # class model): needed by Box/Jackson-Mudholkar OD thresholds and by
        # CI-SIMCA's ccrit. Computed once here, never recomputed on new/test data.
        Xc_train = self.E_train + self.T_train.dot(self.P.T)
        cov_X = np.cov(Xc_train, rowvar=False)
        eig_all = np.linalg.eigvalsh(cov_X)[::-1]
        A = self.pca.n_components_
        residual_eig = eig_all[A:]
        theta1 = float(np.sum(residual_eig))
        theta2 = float(np.sum(residual_eig ** 2))
        theta3 = float(np.sum(residual_eig ** 3))

        # --- OD threshold ---
        if od_method == 'percentile':
            ODcrit = empirical_percentile_threshold(self.OD_train, alpha=self.alpha)

        elif od_method == 'jackson':
            ODcrit = jackson_mudholkar_od_threshold(eig_all, A=A, alpha=self.alpha)

        elif od_method == 'box':
            ODcrit = box_od_threshold(eig_all, A=A, alpha=self.alpha)

        else:
            raise ValueError("Unknown od_method")
        
        # SD threshold
        N = self.T_train.shape[0]
        if sd_method == 'f':
            SDcrit = sd_threshold_hotelling(A, N, alpha=self.alpha)
        elif sd_method == 'chi2':
            SDcrit = sd_threshold_chi2(A, alpha=self.alpha)
        elif sd_method == 'percentile':
            SDcrit = empirical_percentile_threshold(self.SD_train, alpha=self.alpha)
        else:
            raise ValueError("Unknown sd_method")

        self.thresholds = {
                        'ODcrit': float(ODcrit),
                        'SDcrit': float(SDcrit),
                        'od_method': od_method,
                        'sd_method': sd_method,
                        'theta1': theta1,
                        'theta2': theta2,
                        'theta3': theta3,
                        }

        # if dd_params provided, compute DD baseline statistics
        if dd_params is not None:
            # compute OD0, SD0 and their variances for training set
            OD0 = np.mean(self.OD_train)
            SD0 = np.mean(self.SD_train)
            s2_OD = np.var(self.OD_train, ddof=1)
            s2_SD = np.var(self.SD_train, ddof=1)
            self.thresholds.update({
                'OD0': float(OD0),
                'SD0': float(SD0),
                's2_OD': float(s2_OD),
                's2_SD': float(s2_SD),
            })
            self.thresholds['fcrit'] = float(dd_fcrit(OD0, SD0, s2_OD, s2_SD, alpha=self.alpha))

    def predict(self, X_new, rule='sim'):
        """
        Predict membership according to rule:
         - 'sim'  -> Sim-SIMCA: OD <= ODcrit AND SD <= SDcrit
         - 'alt'  -> Alt-SIMCA: d = sqrt((OD/ODc)^2 + (SD/SDc)^2)  accepted if d <= sqrt(2)
         - 'ci'   -> CI-SIMCA: c = OD/ODc + SD/SDc  accepted if c <= ccrit (we use chi2-based ccrit approx)
         - 'dd'   -> DD-SIMCA: f = LOD*(OD/OD0) + LSD*(SD/SD0) accepted if f <= fcrit
        Returns dict with masks and diagnostics arrays.
        """
        X_new = np.asarray(X_new)
        if self.pca is None:
            raise RuntimeError("Model not fitted yet.")

        Xc = X_new - self.mean_
        T_new = self.pca.transform(Xc)
        E_new = Xc - T_new.dot(self.P.T)
        OD_new = orthogonal_distance(E_new)
        SD_new = score_distance(T_new, self.eigvals)

        ODc = self.thresholds.get('ODcrit', np.inf)
        SDc = self.thresholds.get('SDcrit', np.inf)

        if rule == 'sim':
            inside = np.logical_and(OD_new <= ODc, SD_new <= SDc)
            metric = None
        elif rule == 'alt':
            denom = (ODc if ODc > 0 else 1e-12)
            denom2 = (SDc if SDc > 0 else 1e-12)
            d = np.sqrt((OD_new / denom) ** 2 + (SD_new / denom2) ** 2)
            inside = d <= np.sqrt(2.0)
            metric = d
        elif rule == 'ci':
            denom = (ODc if ODc > 0 else 1e-12)
            denom2 = (SDc if SDc > 0 else 1e-12)
            c = (OD_new / denom) + (SD_new / denom2)

            A = self.pca.n_components_
            theta1 = self.thresholds.get('theta1')
            theta2 = self.thresholds.get('theta2')
            if theta1 is None or theta2 is None:
                raise RuntimeError("CI-SIMCA requires set_thresholds() to be called before predict().")

            # Avoid divide by zero - fallback to chi2 with A dof
            if SDc == 0 or ODc == 0:
                ccrit = chi2.ppf(1 - self.alpha, df=A)
            else:
                g = (A / (SDc ** 2) + theta2 / (ODc ** 2)) / (A / SDc + theta1 / ODc + 1e-18)
                h = ((A / SDc + theta1 / ODc) ** 2) / (A / (SDc ** 2) + theta2 / (ODc ** 2) + 1e-18)
                ccrit = g * chi2.ppf(1 - self.alpha, df=max(1, int(round(h))))

            inside = c <= ccrit
            metric = c
        elif rule == 'dd':
            OD0 = self.thresholds.get('OD0', np.mean(self.OD_train))
            SD0 = self.thresholds.get('SD0', np.mean(self.SD_train))
            s2_OD = self.thresholds.get('s2_OD', np.var(self.OD_train, ddof=1))
            s2_SD = self.thresholds.get('s2_SD', np.var(self.SD_train, ddof=1))
            LOD = (2 * (OD0 ** 2)) / (s2_OD + 1e-18)
            LSD = (2 * (SD0 ** 2)) / (s2_SD + 1e-18)
            f_stat = (LOD * (OD_new / OD0)) + (LSD * (SD_new / SD0))
            fcrit = self.thresholds.get('fcrit', dd_fcrit(OD0, SD0, s2_OD, s2_SD, alpha=self.alpha))
            inside = f_stat <= fcrit
            metric = f_stat
        else:
            raise ValueError("Unknown rule")

        if rule == 'sim':
            ratio_OD = OD_new / (ODc if ODc > 0 else 1e-12)
            ratio_SD = SD_new / (SDc if SDc > 0 else 1e-12)
            rule_distance = np.maximum(ratio_OD, ratio_SD) + 0.25 * np.minimum(ratio_OD, ratio_SD)
            crit = None
        elif rule == 'alt':
            rule_distance = metric / np.sqrt(2.0)
            crit = np.sqrt(2.0)
        elif rule == 'ci':
            rule_distance = metric / (ccrit if ccrit > 0 else 1e-12)
            crit = ccrit
        elif rule == 'dd':
            rule_distance = metric / (fcrit if fcrit > 0 else 1e-12)
            crit = fcrit
        else:
            rule_distance = np.zeros_like(OD_new)
            crit = None

        k = np.log(2)
        confidence = 100.0 * np.exp(-k * (rule_distance**2))
        # confidence = 100 * (1 - rule_distance)

        return {
                        'inside': np.array(inside, dtype=bool),
                        'confidence_score': confidence,
                        'OD': OD_new,
                        'SD': SD_new,
                        'metric': metric,
                        'crit': crit,
                        'ODcrit': ODc,
                        'SDcrit': SDc
                        }

# ------------------------
# High-level SIMCA manager (multiple classes)
# ------------------------

class SIMCAModel:
    """
    Manager for multiple class models.
    Usage:
       simca = SIMCAModel(alpha=0.05, default_n_components=3)
       simca.fit(X_train, y_train)
       simca.set_thresholds_all(...)
       preds = simca.predict(X_test, rule='sim')  # returns dict[class] -> results
    """
    def __init__(self, alpha=0.05, default_n_components=None):
        self.alpha = alpha
        self.default_n_components = default_n_components
        self.class_models = {}   # dict label -> SIMCAClassModel

    def fit(self, X, y, n_components=None):
        X = np.asarray(X)
        y = np.asarray(y)
        labels = np.unique(y)
        for lab in labels:
            Xm = X[y == lab]
            cm = SIMCAClassModel(n_components=(n_components if n_components is not None else self.default_n_components),
                                 alpha=self.alpha)
            cm.fit(Xm)
            self.class_models[lab] = cm

    def set_thresholds_all(self, od_method='percentile', sd_method='f', dd_params=None):
        for lab, cm in self.class_models.items():
            cm.set_thresholds(od_method=od_method, sd_method=sd_method, dd_params=dd_params)

    def predict(self, X, rule='sim'):
        out = {}
        for lab, cm in self.class_models.items():
            out[lab] = cm.predict(X, rule=rule)
        return out

    def tune_num_components(
        self,
        X,
        y,
        comp_grid=None,
        cv=5,
        mode='rigorous',
        rule='sim',
        od_method='percentile',
        sd_method='f'
    ):
        """
        Tune number of components per class.
    
        mode:
            'rigorous'  => uses only target class samples to select A
                           (sensitivity close to 1 - alpha)
    
            'compliant' => uses target + non-target to maximize
                           efficiency = sqrt(sens * spec)
    
        Returns
        -------
        selected_A : dict
            Dictionary with selected number of components per class.
        """
    
        X = np.asarray(X)
        y = np.asarray(y)
    
        labels = np.unique(y)
    
        if comp_grid is None:
            comp_grid = list(range(1, min(10, X.shape[1]) + 1))
    
        selected = {}
    
        for lab in labels:
            Xm = X[y == lab]
            Xnon = X[y != lab]
    
            best_A = comp_grid[0]
            best_score = -np.inf
            # Cross-validation on target class
            kf = KFold(
                n_splits=min(cv, max(2, Xm.shape[0])),
                shuffle=True,
                random_state=0
            )
    
            for A in comp_grid:
    
                sens_list = []
                spec_list = []
    
                for train_idx, test_idx in kf.split(Xm):
    
                    Xtr = Xm[train_idx]
                    Xval = Xm[test_idx]
    
                    cm = SIMCAClassModel(
                        n_components=A,
                        alpha=self.alpha
                    )
    
                    cm.fit(Xtr)
                    cm.set_thresholds(
                        od_method=od_method,
                        sd_method=sd_method
                    )
    
                    # Sensitivity on target validation fold
                    res_target = cm.predict(Xval, rule=rule)
                    sens = np.mean(res_target['inside'])
    
                    # Specificity on sampled non-targets
                    n_val = Xval.shape[0]
    
                    if Xnon.shape[0] == 0:
                        spec = 1.0
                    else:
                        rng = np.random.default_rng(42 + A + len(sens_list))
                        idxs = rng.choice(Xnon.shape[0], size=min(Xnon.shape[0], n_val), replace=False)
    
                        non_sample = Xnon[idxs]
    
                        res_non = cm.predict(
                            non_sample,
                            rule=rule
                        )
    
                        # fraction of non-targets rejected
                        spec = np.mean(~res_non['inside'])
    
                    sens_list.append(sens)
                    spec_list.append(spec)
    
                sens_mean = np.mean(sens_list)
                spec_mean = np.mean(spec_list)
    
                if mode == 'rigorous':
                    target_sens = 1.0 - self.alpha
                    score = -abs(sens_mean - target_sens)
    
                    # Look for the sensitivity closest to the target
                    if score > best_score:
                        best_score = score
                        best_A = A
                    # Prefer fewer components on ties (more robust for small N)
                    elif score == best_score and A < best_A:
                        best_A = A
    
                else:  # compliant
                    efficiency = np.sqrt(sens_mean * spec_mean)
    
                    if efficiency > best_score:
                        best_score = efficiency
                        best_A = A
    
            selected[lab] = int(best_A)
    
        return selected


In [3]:
# np.random.seed(0)
# # class A: cluster around 0
# XA = np.random.normal(loc=0.0, scale=1.0, size=(100, 10))
# # class B: cluster around +3 in first two features
# XB = np.random.normal(loc=0.0, scale=1.0, size=(80, 10))
# XB[:,0:2] += 3.0
# X = np.vstack([XA, XB])
# y = np.array([0]*XA.shape[0] + [1]*XB.shape[0])

# simca = SIMCAModel(alpha=0.05)
# simca.fit(X, y)

# # set thresholds (choose methods; you can also tune components first)
# simca.set_thresholds_all(od_method='percentile', sd_method='f')

# # predict on test points
# Xtest = np.vstack([np.mean(XA, axis=0), np.mean(XB, axis=0), np.zeros(10)])
# preds = simca.predict(Xtest, rule='sim')

# for lab, res in preds.items():
#     print("Class", lab, "inside:", res['inside'], "ODcrit", res['ODcrit'], "SDcrit", res['SDcrit'])

In [4]:
# # --- PCA 2D projection ---
# pca2d = PCA(n_components=2)
# X_2d = pca2d.fit_transform(X)
# Xtest_2d = pca2d.transform(Xtest)

# # Define colors for each class (RGB)
# label_colors = {0: np.array([0,0,1]),
#                 1: np.array([1,0,0])}

# plt.figure(figsize=(7,6))

# # Plot original training data
# for lab in np.unique(y):
#     mask = (y == lab)
#     plt.scatter(X_2d[mask,0], X_2d[mask,1], 
#                 c=[label_colors[lab]], label=f"Train {lab}", alpha=0.5, s=50)

# legend_items = {}

# # Plot test points
# for i, vec_2d in enumerate(Xtest_2d):
#     # Find which classes the test point belongs to
#     inside_classes = [lab for lab, res in preds.items() if res['inside'][i]]
#     if len(inside_classes) == 0:
#         # Not inside any class → black cross
#         color = np.array([0,0,0])
#         marker = 'X'
#         label_text = 'None'
#     elif len(inside_classes) == 1:
#         # Inside exactly one class → use that class color
#         color = label_colors[inside_classes[0]]
#         marker = 'X'
#         label_text = f"{inside_classes[0]}"
#     else:
#         # Inside multiple classes → blend colors
#         cols = np.array([label_colors[lab] for lab in inside_classes])
#         color = np.mean(cols, axis=0)
#         marker = 'P'
#         label_text = '+'.join(str(lab) for lab in inside_classes)
    
#     plt.scatter(vec_2d[0], vec_2d[1], c=[color], s=150, marker=marker, edgecolor='k')
#     plt.text(vec_2d[0]+0.05, vec_2d[1]+0.15, f"T{i}", fontsize=9)
    
#     if label_text not in legend_items:
#         legend_items[label_text] = plt.Line2D([0], [0], marker=marker, color='w',
#                                              markerfacecolor=color, markeredgecolor='k', markersize=10)

# plt.legend(list(legend_items.values()), list(legend_items.keys()), title="Test point classes")
# plt.xlabel("PC1")
# plt.ylabel("PC2")
# plt.title("SIMCA classification visualization")
# plt.grid(True)
# plt.show()